# 如何利用sklearn 来做文本分析

### 加载 20 newsgroups 数据集

In [53]:
from sklearn.datasets import fetch_20newsgroups

In [54]:
categories = ['alt.atheism', 'soc.religion.christian',
              'comp.graphics', 'sci.med']

In [55]:
twenty_train = fetch_20newsgroups(subset='train', 
                                  categories=categories, 
                                  shuffle=True, random_state=42)

In [56]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [57]:
len(twenty_train.data)

2257

In [58]:
twenty_train.data[0]

'From: sd345@city.ac.uk (Michael Collier)\nSubject: Converting images to HP LaserJet III?\nNntp-Posting-Host: hampton\nOrganization: The City University\nLines: 14\n\nDoes anyone know of a good way (standard PC application/PD utility) to\nconvert tif/img/tga files into LaserJet III format.  We would also like to\ndo the same, converting to HPGL (HP plotter) files.\n\nPlease email any response.\n\nIs this the correct group?\n\nThanks in advance.  Michael.\n-- \nMichael Collier (Programmer)                 The Computer Unit,\nEmail: M.P.Collier@uk.ac.city                The City University,\nTel: 071 477-8000 x3769                      London,\nFax: 071 477-8565                            EC1V 0HB.\n'

In [59]:
print("\n".join(twenty_train.data[0].split("\n")))

From: sd345@city.ac.uk (Michael Collier)
Subject: Converting images to HP LaserJet III?
Nntp-Posting-Host: hampton
Organization: The City University
Lines: 14

Does anyone know of a good way (standard PC application/PD utility) to
convert tif/img/tga files into LaserJet III format.  We would also like to
do the same, converting to HPGL (HP plotter) files.

Please email any response.

Is this the correct group?

Thanks in advance.  Michael.
-- 
Michael Collier (Programmer)                 The Computer Unit,
Email: M.P.Collier@uk.ac.city                The City University,
Tel: 071 477-8000 x3769                      London,
Fax: 071 477-8565                            EC1V 0HB.



In [60]:
print(twenty_train.target_names[twenty_train.target[0]])

comp.graphics


In [61]:
twenty_train.target[:10]

array([1, 1, 3, 3, 3, 3, 3, 2, 2, 2])

In [62]:
for t in twenty_train.target[:10]:
    print(twenty_train.target_names[t])

comp.graphics
comp.graphics
soc.religion.christian
soc.religion.christian
soc.religion.christian
soc.religion.christian
soc.religion.christian
sci.med
sci.med
sci.med


### 从文本中提取出特征

In [63]:
# 为了使机器学习模型能够工作，我们必须先从文本中提取出一些数值特征。

### 1. Bags of words 模型

Bag-of-words model (BoW model) 最早出现在自然语言处理（Natural Language Processing）和信息检索（Information Retrieval）领域.。该模型忽略掉文本的语法和语序等要素，将其仅仅看作是若干个词汇的集合，文档中每个单词的出现都是独立的。BoW使用一组无序的单词(words)来表达一段文字或一个文档.。

基于文本的BoW模型的一个简单例子如下：

首先给出两个简单的文本文档如下：

> John likes to watch movies. Mary likes too.

> John also likes to watch football games.

基于上述两个文档中出现的单词，构建如下一个词典 (dictionary)：

> {“John”: 1, “likes”: 2,”to”: 3, “watch”: 4, “movies”: 5,”also”: 6, “football”: 7, “games”: 8,”Mary”: 9, “too”: 10}

上面的词典中包含10个单词, 每个单词有唯一的索引, 那么每个文本我们可以使用一个10维的向量来表示。如下：

> [1, 2, 1, 1, 1, 0, 0, 0, 1, 1]

> [1, 1, 1, 1, 0, 1, 1, 1, 0, 0]

该向量与原来文本中单词出现的顺序没有关系，而是词典中每个单词在文本中出现的频率。因此BoW模型可认为是一种统计直方图 (histogram)。

The bags of words representation implies that n_features is the number of distinct words in the corpus: this number is typically larger than 100,000.

If n_samples == 10000, storing X as a NumPy array of type float32 would require 10000 x 100000 x 4 bytes = 4GB in RAM which is barely manageable on today’s computers.

Fortunately, most values in X will be zeros since for a given document less than a few thousand distinct words will be used. For this reason we say that bags of words are typically high-dimensional sparse datasets. We can save a lot of memory by only storing the non-zero parts of the feature vectors in memory.

scipy.sparse matrices are data structures that do exactly this, and scikit-learn has built-in support for these structures.

In [64]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()

In [65]:
X_train_counts = count_vect.fit_transform(twenty_train.data)
X_train_counts.shape

(2257, 35788)

In [66]:
X_train_counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 365886 stored elements and shape (2257, 35788)>

In [67]:
count_vect.vocabulary_.get(u'algorithm')

4690

### 2. TF-IDF Model 模型

TF-IDF（term frequency–inverse document frequency）是一种用于信息检索与数据挖掘的常用加权技术。TF意思是词频(Term Frequency)，IDF意思是逆文本频率指数(Inverse Document Frequency)。

TF-IDF是一种统计方法，用以评估一字词对于一个文件集或一个语料库中的其中一份文件的重要程度。字词的重要性随着它在文件中出现的次数成正比增加，但同时会随着它在语料库中出现的频率成反比下降。TF-IDF加权的各种形式常被搜索引擎应用，作为文件与用户查询之间相关程度的度量或评级。除了TF-IDF以外，因特网上的搜索引擎还会使用基于链接分析的评级方法，以确定文件在搜寻结果中出现的顺序。

在一份给定的文件里，词频（term frequency，TF）指的是某一个给定的词语在该文件中出现的频率。这个数字是对词数(term count)的归一化，以防止它偏向长的文件。（同一个词语在长文件里可能会比短文件有更高的词数，而不管该词语重要与否。）对于在某一特定文件里的词语$t_i$
来说，它的重要性可表示为：

$tf_{i,j} = \frac{n_{i,j}}{\sum_k n_{k,j}}$

以上式子中$𝑛_{𝑖,𝑗}$
是该词$t_i$
在文件$d_j$
中的出现次数，而分母则是在文件$d_j$
中所有字词的出现次数之和。

逆向文件频率（inverse document frequency，IDF）是一个词语普遍重要性的度量。某一特定词语的IDF，可以由总文件数目除以包含该词语之文件的数目，再将得到的商取对数得到：

$idf_i = log \frac{|D|}{|\{ j : t_i \in d_j \}|}$

其中：

|𝐷| :语料库中的文件总数

$|\{  j : t_i \in d_j \}|$: 包含词语$t_i$
的文件数目（即$n_i$,𝑗≠0
的文件数目）如果该词语不在语料库中，就会导致被除数为零，因此一般情况下使用$1 + |\{  j : t_i \in d_j \}|$
然后

$ tfidf_{i, j} = tf_{i,j} \times idf_i $

某一特定文件内的高词语频率，以及该词语在整个文件集合中的低文件频率，可以产生出高权重的TF-IDF。因此，TF-IDF倾向于过滤掉常见的词语，保留重要的词语。

In [68]:
from sklearn.feature_extraction.text import TfidfTransformer

In [69]:
tf_transformer = TfidfTransformer(use_idf=False).fit(X_train_counts)

In [70]:
X_train_tf = tf_transformer.transform(X_train_counts)

In [71]:
X_train_tf.shape

(2257, 35788)

In [72]:
tfidf_transformer = TfidfTransformer()

In [73]:
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)

In [74]:
X_train_tfidf.shape

(2257, 35788)

### 选择模型

In [75]:
from sklearn.naive_bayes import MultinomialNB

In [76]:
clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

In [77]:
docs_new = ['God is love', 'OpenGL on the GPU is fast']

In [78]:
X_new_counts = count_vect.transform(docs_new)

In [79]:
X_new_tfidf = tfidf_transformer.transform(X_new_counts)

In [80]:
predicted = clf.predict(X_new_tfidf)

In [81]:
for doc, category in zip(docs_new, predicted):
    print('%r => %s' % (doc, twenty_train.target_names[category]))

'God is love' => soc.religion.christian
'OpenGL on the GPU is fast' => comp.graphics


### 构建pipeline

In [82]:
from sklearn.pipeline import Pipeline

In [83]:
text_clf = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                    ('clf', MultinomialNB()),
])

In [84]:
text_clf.fit(twenty_train.data, twenty_train.target)

,steps,"[('vect', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


### 模型测试

In [85]:
import numpy as np

In [86]:
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

In [87]:
docs_test = twenty_test.data

In [88]:
predicted = text_clf.predict(docs_test)

In [89]:
np.mean(predicted == twenty_test.target)

np.float64(0.8348868175765646)

In [90]:
from sklearn.linear_model import SGDClassifier

In [91]:
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, random_state=42, max_iter=5, tol=None)),
    ])

In [92]:
text_clf.fit(twenty_train.data, twenty_train.target)

,steps,"[('vect', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [93]:
predicted = text_clf.predict(docs_test)

In [94]:
np.mean(predicted == twenty_test.target)

np.float64(0.9101198402130493)

In [95]:
from sklearn import metrics

In [96]:
print(metrics.classification_report(twenty_test.target, predicted, target_names=twenty_test.target_names))

                        precision    recall  f1-score   support

           alt.atheism       0.95      0.80      0.87       319
         comp.graphics       0.87      0.98      0.92       389
               sci.med       0.94      0.89      0.91       396
soc.religion.christian       0.90      0.95      0.93       398

              accuracy                           0.91      1502
             macro avg       0.91      0.91      0.91      1502
          weighted avg       0.91      0.91      0.91      1502



In [97]:
metrics.confusion_matrix(twenty_test.target, predicted)

array([[256,  11,  16,  36],
       [  4, 380,   3,   2],
       [  5,  35, 353,   3],
       [  5,  11,   4, 378]])

### 利用grid search 来进行参数优化

In [98]:
from sklearn.model_selection import GridSearchCV

In [99]:
parameters = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
    'clf__alpha': (1e-2, 1e-3),
}

In [100]:
gs_clf = GridSearchCV(text_clf, parameters, cv=5, n_jobs=-1)

In [101]:
gs_clf = gs_clf.fit(twenty_train.data[:400], twenty_train.target[:400])

In [102]:
twenty_train.target_names[gs_clf.predict(['God is love'])[0]]

'soc.religion.christian'

In [103]:
gs_clf.best_score_

for param_name in sorted(parameters.keys()):
    print("%s: %r" % (param_name, gs_clf.best_params_[param_name]))

clf__alpha: 0.001
tfidf__use_idf: True
vect__ngram_range: (1, 1)


In [104]:
gs_clf.cv_results_

{'mean_fit_time': array([0.39895601, 1.60517917, 0.58846426, 1.69413886, 1.01898074,
        1.49690514, 0.5115231 , 0.93850684]),
 'std_fit_time': array([0.08227762, 0.47214998, 0.35518849, 0.67445955, 0.10257317,
        0.20547506, 0.0531508 , 0.10799278]),
 'mean_score_time': array([0.09107814, 0.27627096, 0.12192311, 0.23230672, 0.12248368,
        0.1150002 , 0.102597  , 0.09302077]),
 'std_score_time': array([0.06099725, 0.15932278, 0.07438434, 0.13138114, 0.02049769,
        0.01379639, 0.02348704, 0.01234505]),
 'param_clf__alpha': masked_array(data=[0.01, 0.01, 0.01, 0.01, 0.001, 0.001, 0.001, 0.001],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=1e+20),
 'param_tfidf__use_idf': masked_array(data=[True, True, False, False, True, True, False, False],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=True),
 'param_vect__ngram_range': masked_array(data=[(1, 1), (1, 2), (1, 1), (1, 2)